# Code to calculate the weighted average velocity for each month

## import libraries

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates
from tqdm.auto import tqdm
import os
from pathlib import Path

/Users/lindsaysummers/micromamba/envs/itslive_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## modify inputs

In [2]:
#start date and end date (string) (YYYY-MM-DD)
start = '2015-01-01'
end = '2026-01-01'

#path to folder with velocities
velocity_path = Path('/Users/lindsaysummers/Documents/Research/Alaska_seasonality/itslive_velocities/')

#path to folder to store outputs
output_path = Path('/Users/lindsaysummers/Documents/Research/Alaska_seasonality/monthly_velocities')

## store velocity csvs in a dictionary

In [3]:
# velocities = {}

# for file in velocity_path.glob("*.csv"):
#     rgi_id = file.stem
#     df = pd.read_csv(file)
#     velocities[rgi_id] = df

velocities = {}

for file in velocity_path.glob("*/*.csv"):
    # RGI folder name
    rgi_id = file.parent.name
    df = pd.read_csv(file)
    velocities[rgi_id] = df

print("Number of RGIs:", len(velocities))
print("RGI IDs:", list(velocities.keys()))

for rgi_id, df in velocities.items():
    print(rgi_id, df.shape)

Number of RGIs: 60
RGI IDs: ['RGI2000-v7.0-G-01-07671', 'RGI2000-v7.0-G-02-04567', 'RGI2000-v7.0-G-02-08538', 'RGI2000-v7.0-G-01-27011', 'RGI2000-v7.0-G-02-10336', 'RGI2000-v7.0-G-01-02509', 'RGI2000-v7.0-G-02-04482', 'RGI2000-v7.0-G-01-04948', 'RGI2000-v7.0-G-01-02613', 'RGI2000-v7.0-G-01-27357', 'RGI2000-v7.0-G-01-09262', 'RGI2000-v7.0-G-02-04479', 'RGI2000-v7.0-G-01-12416', 'RGI2000-v7.0-G-01-04706', 'RGI2000-v7.0-G-01-06051', 'RGI2000-v7.0-G-01-03986', 'RGI2000-v7.0-G-01-17115', 'RGI2000-v7.0-G-01-13344', 'RGI2000-v7.0-G-01-17373', 'RGI2000-v7.0-G-02-05777', 'RGI2000-v7.0-G-01-08766', 'RGI2000-v7.0-G-02-10031', 'RGI2000-v7.0-G-01-18905', 'RGI2000-v7.0-G-01-11331', 'RGI2000-v7.0-G-01-07898', 'RGI2000-v7.0-G-01-05906', 'RGI2000-v7.0-G-02-06848', 'RGI2000-v7.0-G-01-01595', 'RGI2000-v7.0-G-01-16980', 'RGI2000-v7.0-G-01-21534', 'RGI2000-v7.0-G-01-27210', 'RGI2000-v7.0-G-02-08338', 'RGI2000-v7.0-G-01-11320', 'RGI2000-v7.0-G-01-09340', 'RGI2000-v7.0-G-01-06338', 'RGI2000-v7.0-G-01-20403',

## create time "buckets" for each month

In [4]:
#create buckets of time for each month
month_intervals = pd.date_range(start=start,end=end,freq='MS')

month_intervals_list = []

for i in range(len(month_intervals) - 1):
    bucket_start = month_intervals[i]
    bucket_end = month_intervals[i + 1] - pd.Timedelta(days=1)
    month_intervals_list.append({'start': bucket_start,'end': bucket_end})

month_buckets = pd.DataFrame(month_intervals_list)

print(month_buckets)

         start        end
0   2015-01-01 2015-01-31
1   2015-02-01 2015-02-28
2   2015-03-01 2015-03-31
3   2015-04-01 2015-04-30
4   2015-05-01 2015-05-31
..         ...        ...
127 2025-08-01 2025-08-31
128 2025-09-01 2025-09-30
129 2025-10-01 2025-10-31
130 2025-11-01 2025-11-30
131 2025-12-01 2025-12-31

[132 rows x 2 columns]


In [5]:
filtered_velocities = {}

for bucket_index, bucket in month_buckets.iterrows(): #iterate through rows of df with month buckets
    bucket_start = pd.to_datetime(bucket['start']) #change start and end dates to datetime format
    bucket_end = pd.to_datetime(bucket['end'])
    filtered_velocities[bucket_index] = {} #place to store buckets with velocity data
    for rgi_id, velocity_df in velocities.items(): #loop through each RGI ID
        velocity_df['mid_date'] = pd.to_datetime(velocity_df['mid_date']) #convert to datetime format
        filtered_df = velocity_df.loc[(velocity_df['mid_date'] >= bucket_start) & (velocity_df['mid_date'] <= bucket_end)].copy() #filter vel data to current month's date range
        filtered_velocities[bucket_index][rgi_id] = filtered_df #store using RGI ID as dictionary key

print("Filtered velocity data for each bucket and RGI ID")

Filtered velocity data for each bucket and RGI ID


## weight and average velocities for each month/bucket

In [6]:
weighted_avg_velocity = {}

for bucket_index, bucket in month_buckets.iterrows():

    bucket_start = pd.to_datetime(bucket['start'])
    bucket_end = pd.to_datetime(bucket['end'])

    weighted_avg_velocity[bucket_index] = {}

    #loop through each RGI
    for rgi_id, velocity_df in velocities.items():
        velocity_df = velocity_df.copy()

        #convert mid_date to datetime
        velocity_df['mid_date'] = pd.to_datetime(velocity_df['mid_date'])

        #make sure duration is numeric
        velocity_df['date_dt_days'] = pd.to_numeric(velocity_df['date_dt_days'],errors='coerce')
        
        #start and end dates for each itslive calculation
        velocity_df['start_date'] = (velocity_df['mid_date'] - pd.to_timedelta(velocity_df['date_dt_days'] / 2,unit='D'))

        velocity_df['end_date'] = (velocity_df['mid_date'] + pd.to_timedelta(velocity_df['date_dt_days'] / 2,unit='D'))

        #find observations that overlap current month
        itslive_bucket_data = velocity_df.loc[(velocity_df['start_date'] <= bucket_end) &(velocity_df['end_date'] >= bucket_start)].copy()

        if itslive_bucket_data.empty:
            continue

        #calculate overlap with calendar month
        overlap_start = itslive_bucket_data['start_date'].clip(lower=bucket_start)

        overlap_end = itslive_bucket_data['end_date'].clip(upper=bucket_end)

        overlap_days = (overlap_end - overlap_start).dt.total_seconds() / (24 * 60 * 60)

        overlap_days = overlap_days.clip(lower=0)

        itslive_bucket_data['overlap_days'] = overlap_days

        #remove observations with nans
        valid_data = itslive_bucket_data.dropna(subset=['velocity_mean']).copy()

        valid_data = valid_data[valid_data['overlap_days'] > 0]

        if valid_data.empty:
            continue

        #calculate weighted average for each point
        point_results = {}

        for (segment_id, point_id), point_data in valid_data.groupby(['segment_id', 'point_id']):

            total_weighted_velocity = (point_data['velocity_mean'] * point_data['overlap_days']).sum()

            total_days = point_data['overlap_days'].sum()

            if total_days > 0:
                weighted_average = (total_weighted_velocity /total_days)

            else:
                weighted_average = np.nan

            point_results[(segment_id, point_id)] = weighted_average

        weighted_avg_velocity[bucket_index][rgi_id] = point_results

In [7]:
#convert to a df
weighted_avg_rows = []

for bucket_index, rgi_dict in weighted_avg_velocity.items():
    month_start = month_buckets.loc[bucket_index, 'start']
    for rgi_id, point_dict in rgi_dict.items():
        for (segment_id, point_id), velocity in point_dict.items():
            weighted_avg_rows.append({
                'month': month_start,
                'rgi_id': rgi_id,
                'segment_id': segment_id,
                'point_id': point_id,
                'velocity_mean': velocity
            })


weighted_avg_df = pd.DataFrame(weighted_avg_rows)

weighted_avg_df = weighted_avg_df.sort_values(['rgi_id', 'segment_id', 'point_id', 'month']).reset_index(drop=True)

print(weighted_avg_df.head())

       month                   rgi_id  segment_id  point_id  velocity_mean
0 2015-01-01  RGI2000-v7.0-G-01-01595           0         1      57.222220
1 2015-02-01  RGI2000-v7.0-G-01-01595           0         1      56.661233
2 2015-03-01  RGI2000-v7.0-G-01-01595           0         1      40.200572
3 2015-04-01  RGI2000-v7.0-G-01-01595           0         1      73.259594
4 2015-05-01  RGI2000-v7.0-G-01-01595           0         1      27.318688


In [8]:
#check
weighted_avg_df[
    (weighted_avg_df['rgi_id'] == 'RGI2000-v7.0-G-01-01595') &
    (weighted_avg_df['segment_id'] == 0) &
    (weighted_avg_df['point_id'] == 5)
]

,month,rgi_id,segment_id,point_id,velocity_mean
452,2015-01-01,RGI2000-v7.0-G-01-01595,0,5,33.333332
453,2015-02-01,RGI2000-v7.0-G-01-01595,0,5,37.400496
454,2015-03-01,RGI2000-v7.0-G-01-01595,0,5,51.142054
455,2015-04-01,RGI2000-v7.0-G-01-01595,0,5,50.331179
456,2015-05-01,RGI2000-v7.0-G-01-01595,0,5,84.378627
...,...,...,...,...,...
561,2024-09-01,RGI2000-v7.0-G-01-01595,0,5,103.250000
562,2024-10-01,RGI2000-v7.0-G-01-01595,0,5,62.269475
563,2024-11-01,RGI2000-v7.0-G-01-01595,0,5,46.505510
564,2024-12-01,RGI2000-v7.0-G-01-01595,0,5,46.888203


## export

In [9]:
#create main output folder if it doesn't exist
output_path.mkdir(parents=True, exist_ok=True)


#save one CSV for each RGI
for rgi_id, glacier_df in weighted_avg_df.groupby('rgi_id'):

    #create folder for current RGI
    rgi_folder = output_path / rgi_id
    rgi_folder.mkdir(parents=True, exist_ok=True)

    #sort data
    glacier_df = glacier_df.sort_values(['segment_id', 'point_id', 'month'])

    #output filename
    output_file = (rgi_folder /f'{rgi_id}_velocity_monthly.csv')

    #save
    glacier_df.to_csv(output_file,index=False)

    print(f'Saved: {output_file}')

Saved: /Users/lindsaysummers/Documents/Research/Alaska_seasonality/monthly_velocities/RGI2000-v7.0-G-01-01595/RGI2000-v7.0-G-01-01595_velocity_monthly.csv
Saved: /Users/lindsaysummers/Documents/Research/Alaska_seasonality/monthly_velocities/RGI2000-v7.0-G-01-01751/RGI2000-v7.0-G-01-01751_velocity_monthly.csv
Saved: /Users/lindsaysummers/Documents/Research/Alaska_seasonality/monthly_velocities/RGI2000-v7.0-G-01-02509/RGI2000-v7.0-G-01-02509_velocity_monthly.csv
Saved: /Users/lindsaysummers/Documents/Research/Alaska_seasonality/monthly_velocities/RGI2000-v7.0-G-01-02613/RGI2000-v7.0-G-01-02613_velocity_monthly.csv
Saved: /Users/lindsaysummers/Documents/Research/Alaska_seasonality/monthly_velocities/RGI2000-v7.0-G-01-03986/RGI2000-v7.0-G-01-03986_velocity_monthly.csv
Saved: /Users/lindsaysummers/Documents/Research/Alaska_seasonality/monthly_velocities/RGI2000-v7.0-G-01-04706/RGI2000-v7.0-G-01-04706_velocity_monthly.csv
Saved: /Users/lindsaysummers/Documents/Research/Alaska_seasonality/mon